In [19]:
import pandas as pd
import numpy as np

np.random.seed(42)
num_patients = 250000

# Realistic Age Distribution
ages = np.clip(np.round(np.random.normal(loc=48, scale=16, size=num_patients)), 18, 95).astype(int)

# Sex
sexes = np.random.choice(['Male', 'Female'], num_patients, p=[0.51, 0.49])
is_male = (sexes == 'Male').astype(int)

# Realistic BMI Distribution
bmis = np.clip(np.round(np.random.normal(loc=28.5, scale=5.2, size=num_patients), 1), 15, 55)

# Enhanced Blood Pressure
systolic_bp_base = 100 + 0.45 * ages + np.where(bmis < 18.5, 5, 0.8 * bmis) + is_male * 6 + np.random.normal(0, 5, num_patients) # Add a small boost for underweight with potential higher BP
systolic_bps = np.clip(np.round(systolic_bp_base + np.random.normal(0, 15, num_patients)), 90, 220).astype(int)
diastolic_bp_base = 60 + 0.3 * ages + np.where(bmis < 18.5, 3, 0.5 * bmis) + is_male * 3 + np.random.normal(0, 3, num_patients) # Add a small boost for underweight with potential higher BP
diastolic_bps = np.clip(np.round(diastolic_bp_base + np.random.normal(0, 10, num_patients)), 50, 140).astype(int)

# Alcohol Consumption
alcohol_options = ['None', 'Light', 'Moderate', 'Heavy']
alcohol_probs_base = np.array([0.4, 0.35, 0.2, 0.05])
alcohol_probs_adjusted = alcohol_probs_base + np.array([(0) - (0),
                                                      (0) - (0),
                                                      (0),
                                                      (0)])
alcohol_probs_final = np.clip(alcohol_probs_adjusted, 0, 1)
alcohol_probs_final = alcohol_probs_final / np.sum(alcohol_probs_final, axis=0)
alcohol_consumption = np.random.choice(alcohol_options, num_patients, p=alcohol_probs_final)

# Diet Quality
diet_options = ['Poor', 'Fair', 'Good', 'Excellent']
diet_probs_base = np.array([0.35, 0.4, 0.2, 0.05])
ses_proxy = np.random.rand(num_patients)
diet_probs_adjustment = np.array([
    0.05 * (1 - ses_proxy),
    -0.025 * (1 - ses_proxy),
    -0.015 * ses_proxy,
    -0.01 * ses_proxy
])
diet_probs_adjusted = diet_probs_base[:, np.newaxis] + diet_probs_adjustment
diet_probs_clipped = np.clip(diet_probs_adjusted, 0, 1)
diet_probs_final = diet_probs_clipped / np.sum(diet_probs_clipped, axis=0, keepdims=True)
diet_quality = np.array([np.random.choice(diet_options, p=prob) for prob in diet_probs_final.T])

# Family History
family_history = np.random.choice([0, 1], num_patients, p=[0.3, 0.7])

# Enhanced Cholesterol (indirect link to BMI)
total_cholesterol_base = 180 + 0.4 * ages + np.random.normal(0, 15, num_patients) + (diet_quality == 'Poor') * 20 - (diet_quality == 'Excellent') * 10 + family_history * 15 + (bmis > 30) * 5 - (bmis < 20) * 2 # Weak link to BMI
total_cholesterols = np.clip(np.round(total_cholesterol_base + np.random.normal(0, 35, num_patients)), 100, 400).astype(int)
hdl_cholesterol_base = 55 - 0.15 * ages - (diet_quality == 'Poor') * 10 + (diet_quality == 'Excellent') * 8 - family_history * 5 + is_male * 3 - (bmis > 30) * 2 + (bmis < 20) * 1 # Weak inverse link to BMI
hdl_cholesterols = np.clip(np.round(hdl_cholesterol_base + np.random.normal(0, 12, num_patients)), 20, 100).astype(int)
triglyceride_base = 80 + 0.6 * ages + (diet_quality == 'Poor') * 40 + (alcohol_consumption == 'Heavy') * 30 - (diet_quality == 'Excellent') * 15 + np.random.normal(0, 20, num_patients) + (bmis > 30) * 8 - (bmis < 20) * 3 # Weak positive link to BMI
triglycerides = np.clip(np.round(triglyceride_base + np.random.normal(0, 50, num_patients)), 50, 500).astype(int)

# Triglycerides
triglyceride_base = 80 + 0.6 * ages + (diet_quality == 'Poor') * 40 + (alcohol_consumption == 'Heavy') * 30 - (diet_quality == 'Excellent') * 15 + np.random.normal(0, 20, num_patients)
triglycerides = np.clip(np.round(triglyceride_base + np.random.normal(0, 50, num_patients)), 50, 500).astype(int)

# LDL Cholesterol
ldl_cholesterols = np.clip(np.round(total_cholesterols - hdl_cholesterols - (triglycerides / 5)), 50, 300).astype(int)
ldl_cholesterols[ldl_cholesterols < 0] = 50

# Physical Activity
physical_activity_base = 150 - 1.5 * bmis - (ages - 30) / 3 * 5 - (diet_quality == 'Poor') * 20 + \
                        (diet_quality == 'Excellent') * 30 + 10 * ses_proxy
physical_activity = np.clip(np.round(physical_activity_base + np.random.normal(0, 50, num_patients)), 0, 300).astype(int)

# Smoking
smoking_prob_base = 0.05 + 0.3 * np.exp(-((ages - 35) ** 2) / (2 * 20 ** 2)) + \
                    (diet_quality == 'Poor') * 0.1 + (alcohol_consumption == 'Heavy') * 0.08 - \
                    (diet_quality == 'Excellent') * 0.05 - 0.05 * ses_proxy
smoking_prob = np.clip(smoking_prob_base, 0, 0.7)
smoking = (np.random.rand(num_patients) < smoking_prob).astype(int)

# Diabetes
diabetes = np.zeros(num_patients, dtype=int)
diabetes_prob_type2_base = 0.005 + (ages - 20) / 150 * 0.3 + (bmis - 23) / 15 * 0.4 + \
                           family_history * 0.1 + (diet_quality == 'Poor') * 0.15 - \
                           (physical_activity > 150) * 0.08
has_diabetes_type2 = (np.random.rand(num_patients) < np.clip(diabetes_prob_type2_base, 0, 0.6))

diabetes_prob_type1_base = 0.003 + np.exp(-((ages - 12) ** 2) / (2 * 10 ** 2)) * 0.05 + family_history * 0.02
has_diabetes_type1 = (np.random.rand(num_patients) < np.clip(diabetes_prob_type1_base, 0, 0.15))

diabetes[has_diabetes_type2] = 2
diabetes[has_diabetes_type1 & ~has_diabetes_type2] = 1

# Anemia
anemia_prob_base = 0.05 + kidney_disease * 0.4 - (diet_quality == 'Excellent') * 0.05 + (ages > 65) * 0.1
anemia = (np.random.rand(num_patients) < np.clip(anemia_prob_base, 0, 0.4)).astype(int)

# Initial estimation of Sleep Duration (without stress_level)
sleep_duration_base_initial = 7 - (ages > 65) * 0.5 + (physical_activity > 100) * 0.3 + 0.2 * ses_proxy
sleep_duration_initial = np.clip(np.round(sleep_duration_base_initial + np.random.normal(0, 1.2, num_patients)), 4, 10)

# Stress Level (using the initial sleep_duration)
stress_level_base_initial = 5 + 1.5 * (1 - ses_proxy) - (sleep_duration_initial > 7) * 1 + (physical_activity > 100) * -0.5
stress_level = np.clip(np.round(stress_level_base_initial + np.random.normal(0, 2, num_patients)), 1, 10).astype(int)

# Refined Sleep Duration (using the calculated stress_level)
sleep_duration_base = 7 - 0.5 * stress_level - (ages > 65) * 0.5 + (physical_activity > 100) * 0.3 + 0.2 * ses_proxy
sleep_duration = np.clip(np.round(sleep_duration_base + np.random.normal(0, 1.2, num_patients)), 4, 10)

# Heart Rate
heart_rate_base = 70 + 0.2 * ages + 0.4 * stress_level - 0.1 * physical_activity + 0.1 * bmis # Uses ages, stress_level, physical_activity, bmis
heart_rate = np.clip(np.round(heart_rate_base + np.random.normal(0, 10, num_patients)), 40, 120).astype(int)

# ECG Abnormality
ecg_abnormality_prob_base = 0.02 + (ages - 50) / 120 * 0.3 + previous_heart_problems * 0.4 + \
                             (diabetes > 0) * 0.2 + (systolic_bps > 140) * 0.15 # Note: previous_heart_problems is defined later
ecg_abnormality = (np.random.rand(num_patients) < np.clip(ecg_abnormality_prob_base, 0, 0.6)).astype(int)

# Previous Heart Problems
previous_heart_problems_prob_base = 0.01 + (ages - 40) / 150 * 0.4 + (diabetes > 0) * 0.3 + \
                                   (systolic_bps > 140) * 0.2 + family_history * 0.15
previous_heart_problems = (np.random.rand(num_patients) < np.clip(previous_heart_problems_prob_base, 0, 0.7)).astype(int)

# Kidney Disease
kidney_disease_prob_base = 0.008 + (diabetes > 0) * 0.4 + (systolic_bps > 140) * 0.25 + \
                           (ages > 50) * 0.2 + 0.05 * (1 - ses_proxy)
kidney_disease = (np.random.rand(num_patients) < np.clip(kidney_disease_prob_base, 0, 0.5)).astype(int)

# Heart Disease Risk (Enhanced for low BMI)
heart_disease_risk_base = (ages - 30) / 50 * 0.25 + \
                          (np.where(bmis < 18.5, -0.05, (bmis - 22) / 20) * 0.2) + \
                          (systolic_bps - 110) / 80 * 0.15 + \
                          (ldl_cholesterols - 100) / 80 * 0.12 - \
                          (hdl_cholesterols - 40) / 20 * 0.08 + \
                          triglycerides / 400 * 0.05 + \
                          smoking * 0.25 + \
                          (diabetes > 0) * 0.35 + \
                          family_history * 0.18 + \
                          previous_heart_problems * 0.45 + \
                          kidney_disease * 0.25 + \
                          ecg_abnormality * 0.3 + \
                          (physical_activity < 60) * 0.05 + \
                          (diet_quality == 'Poor') * 0.08 + \
                          ((bmis < 17) & (kidney_disease == 1)) * 0.1 + \
                          ((bmis < 17) & (anemia == 1)) * 0.08
heart_disease_prob = np.clip(heart_disease_risk_base, 0, 0.9)
heart_disease = (np.random.rand(num_patients) < heart_disease_prob).astype(int)

# Heart Failure Risk (Enhanced for low BMI)
heart_failure_risk_base = (ages - 55) / 30 * 0.4 + \
                          (np.where(bmis < 18.5, -0.03, (bmis - 27) / 15) * 0.15) + \
                          heart_disease * 0.7 + \
                          previous_heart_problems * 0.5 + \
                          kidney_disease * 0.3 + \
                          ecg_abnormality * 0.4 + \
                          (diabetes > 0) * 0.25 + \
                          (systolic_bps > 150) * 0.1 + \
                          (ldl_cholesterols > 130) * 0.05 + \
                          anemia * 0.08 + \
                          ((bmis < 17) & (kidney_disease == 1)) * 0.15 + \
                          ((bmis < 17) & (anemia == 1)) * 0.1
heart_failure_prob = np.clip(heart_failure_risk_base, 0, 0.7)
heart_failure = (np.random.rand(num_patients) < heart_failure_prob).astype(int)
heart_failure = np.logical_and(heart_failure, heart_disease).astype(int)

# Medication Usage
medication_hypertension = (((systolic_bps > 140) | (diastolic_bps > 90)) * 0.7 +
                           (diabetes > 0) * 0.4 +
                           (ages > 65) * 0.5 +
                           (previous_heart_problems == 1) * 0.3 +
                           (np.random.rand(num_patients) < 0.3)).astype(int)

medication_cholesterol = ((ldl_cholesterols > 130) * 0.8 +
                          family_history * 0.4 +
                          (ages > 50) * 0.5 +
                          (previous_heart_problems == 1) * 0.4 +
                          (diabetes > 0) * 0.3 +
                          (np.random.rand(num_patients) < 0.25)).astype(int)

# Create DataFrame
data = pd.DataFrame({
    'PatientID': np.arange(1, num_patients + 1),
    'Age': ages,
    'Sex': sexes,
    'BMI': bmis,
    'SystolicBP': systolic_bps,
    'DiastolicBP': diastolic_bps,
    'Cholesterol': total_cholesterols,
    'HDL_Cholesterol': hdl_cholesterols,
    'LDL_Cholesterol': ldl_cholesterols,
    'Triglycerides': triglycerides,
    'Smoking': smoking,
    'Diabetes': diabetes,
    'FamilyHistory': family_history,
    'PhysicalActivity': physical_activity,
    'AlcoholConsumption': alcohol_consumption,
    'DietQuality': diet_quality,
    'PreviousHeartProblems': previous_heart_problems,
    'KidneyDisease': kidney_disease,
    'Anemia': anemia,
    'StressLevel': stress_level,
    'SleepDuration': sleep_duration,
    'HeartRate': heart_rate,
    'ECG_Abnormality': ecg_abnormality,
    'Medication_Hypertension': medication_hypertension,
    'Medication_Cholesterol': medication_cholesterol,
    'Heart Failure': heart_failure,
    'Heart Disease': heart_disease
})

# Save to CSV
data.to_csv('realistic_synthetic_heart_data_enhanced.csv', index=False)
print(f"Generated an enhanced realistic synthetic dataset of {num_patients} rows and saved it to 'realistic_synthetic_heart_data_enhanced.csv'")

Generated an enhanced realistic synthetic dataset of 250000 rows and saved it to 'realistic_synthetic_heart_data_enhanced.csv'


In [21]:
import pandas as pd
import numpy as np

np.random.seed(42)
num_patients = 250000

# Realistic Age Distribution (with slight skew towards older ages)
ages = np.clip(np.round(np.random.beta(a=2, b=3, size=num_patients) * 77 + 18), 18, 95).astype(int)

# Sex
sexes = np.random.choice(['Male', 'Female'], num_patients, p=[0.51, 0.49])
is_male = (sexes == 'Male').astype(int)

# Realistic BMI Distribution (with heavier right tail)
bmis = np.clip(np.round(np.random.lognormal(mean=3.2, sigma=0.2, size=num_patients), 1), 15, 60)

# Enhanced Blood Pressure (Non-linear BMI effect, interaction with age)
systolic_bp_base = 100 + 0.5 * ages + 0.01 * ages**2 + (0.7 * bmis + 0.005 * bmis**2) + is_male * 5 + np.random.normal(0, 10, num_patients)
systolic_bps = np.clip(np.round(systolic_bp_base), 90, 250).astype(int)
diastolic_bp_base = 60 + 0.3 * ages + (0.4 * bmis + 0.003 * bmis**2) + is_male * 2 + np.random.normal(0, 7, num_patients)
diastolic_bps = np.clip(np.round(diastolic_bp_base), 50, 150).astype(int)

# Alcohol Consumption (More realistic probabilities based on age and sex)
alcohol_options = ['None', 'Light', 'Moderate', 'Heavy']
alcohol_probs = np.zeros((num_patients, len(alcohol_options)))
for i in range(num_patients):
    age_factor = (ages[i] - 40) / 30
    sex_factor = 0.1 if sexes[i] == 'Male' else -0.05
    base_probs = np.array([0.4, 0.35, 0.2, 0.05])
    adjustment = np.array([-0.1 - sex_factor, -0.05 + sex_factor, 0.1 + age_factor, 0.05 + age_factor])
    final_probs = np.clip(base_probs + adjustment, 0.01, 0.98)
    alcohol_probs[i] = final_probs / np.sum(final_probs)
alcohol_consumption = np.array([np.random.choice(alcohol_options, p=prob) for prob in alcohol_probs])

# Diet Quality (Influenced by SES and age)
diet_options = ['Poor', 'Fair', 'Good', 'Excellent']
ses_proxy = np.random.beta(a=1.5, b=3.5, size=num_patients) # Skewed towards lower SES
diet_probs = np.zeros((num_patients, len(diet_options)))
for i in range(num_patients):
    age_factor = (ages[i] - 50) / 40
    base_probs = np.array([0.35, 0.4, 0.2, 0.05])
    adjustment = np.array([0.05 * (1 - ses_proxy[i]) + 0.05 * age_factor,
                           -0.025 * (1 - ses_proxy[i]) - 0.02 * age_factor,
                           -0.015 * ses_proxy[i] - 0.01 * age_factor,
                           -0.01 * ses_proxy[i] + 0.03 * age_factor])
    final_probs = np.clip(base_probs + adjustment, 0.01, 0.98)
    diet_probs[i] = final_probs / np.sum(final_probs)
diet_quality = np.array([np.random.choice(diet_options, p=prob) for prob in diet_probs])
# Family History (Age-dependent)
family_history_prob = 0.2 + 0.3 * (ages / 95)
family_history = (np.random.rand(num_patients) < family_history_prob).astype(int)

# Enhanced Cholesterol (Non-linear, interactions with diet, BMI, family history)
total_cholesterol_base = 160 + 0.6 * ages + 0.008 * ages**2 + (diet_quality == 'Poor') * 30 - (diet_quality == 'Excellent') * 15 + family_history * 20 + 0.3 * bmis + 0.002 * bmis**2 + np.random.normal(0, 25, num_patients)
total_cholesterols = np.clip(np.round(total_cholesterol_base), 100, 450).astype(int)
hdl_cholesterol_base = 60 - 0.2 * ages - (diet_quality == 'Poor') * 12 + (diet_quality == 'Excellent') * 10 - family_history * 8 - 0.15 * bmis + is_male * 5 + np.random.normal(0, 10, num_patients)
hdl_cholesterols = np.clip(np.round(hdl_cholesterols), 20, 120).astype(int)
triglyceride_base = 70 + 0.8 * ages + (diet_quality == 'Poor') * 50 + (alcohol_consumption == 'Heavy') * 40 - (diet_quality == 'Excellent') * 20 + 0.5 * bmis + np.random.normal(0, 40, num_patients)
triglycerides = np.clip(np.round(triglyceride_base), 50, 600).astype(int)
ldl_cholesterols = np.clip(np.round(total_cholesterols - hdl_cholesterols - (triglycerides / 5)), 30, 350).astype(int)
ldl_cholesterols[ldl_cholesterols < 0] = 30

# Physical Activity (Non-linear with BMI and age, SES influence)
physical_activity_base = 200 - 2 * bmis + 0.02 * bmis**2 - 1.2 * ages + 0.01 * ages**2 + 50 * ses_proxy + np.random.normal(0, 60, num_patients)
physical_activity = np.clip(np.round(physical_activity_base), 0, 400).astype(int)

# Smoking (Age, SES, Alcohol, Diet dependent probability)
smoking_prob_base = 0.03 + 0.4 * np.exp(-((ages - 30) ** 2) / (2 * 15 ** 2)) + (diet_quality == 'Poor') * 0.15 + (alcohol_consumption == 'Heavy') * 0.1 - (diet_quality == 'Excellent') * 0.08 - 0.1 * ses_proxy
smoking_prob = np.clip(smoking_prob_base, 0, 0.8)
smoking = (np.random.rand(num_patients) < smoking_prob).astype(int)

# Diabetes (Type 1 and Type 2 with different age dependencies and risk factors)
diabetes = np.zeros(num_patients, dtype=int)
diabetes_prob_type2_base = 0.002 + (ages - 40) / 100 * 0.4 + (bmis - 25) / 10 * 0.5 + family_history * 0.15 + (diet_quality == 'Poor') * 0.2 - (physical_activity > 150) * 0.1
has_diabetes_type2 = (np.random.rand(num_patients) < np.clip(diabetes_prob_type2_base, 0, 0.7))
diabetes_prob_type1_base = 0.005 + np.exp(-((ages - 15) ** 2) / (2 * 8 ** 2)) * 0.08 + family_history * 0.03
has_diabetes_type1 = (np.random.rand(num_patients) < np.clip(diabetes_prob_type1_base, 0, 0.2))
diabetes[has_diabetes_type2] = 2 # Type 2
diabetes[has_diabetes_type1 & ~has_diabetes_type2] = 1 # Type 1

# Anemia (Age and Kidney Disease dependent)
anemia_prob_base = 0.03 + kidney_disease * 0.5 + (ages > 60) * 0.12 - (diet_quality == 'Excellent') * 0.03
anemia = (np.random.rand(num_patients) < np.clip(anemia_prob_base, 0, 0.5)).astype(int)

# Sleep Duration (Stress, Age, Physical Activity, SES dependent)
sleep_duration_base = 7.5 - 0.3 * stress_level - 0.01 * ages + 0.0001 * ages**2 + 0.2 * (physical_activity > 100) - 0.1 * (1 - ses_proxy) + np.random.normal(0, 1.5, num_patients)
sleep_duration = np.clip(np.round(sleep_duration_base), 4, 12).astype(int)

# Stress Level (Sleep, SES, Physical Activity dependent)
stress_level_base = 6 + 0.8 * (1 - ses_proxy) - 0.2 * sleep_duration - 0.005 * physical_activity + np.random.normal(0, 2.5, num_patients)
stress_level = np.clip(np.round(stress_level_base), 1, 10).astype(int)

# Heart Rate (Age, Stress, Physical Activity, BMI dependent)
heart_rate_base = 75 + 0.15 * ages + 0.3 * stress_level - 0.08 * physical_activity + 0.2 * bmis + np.random.normal(0, 12, num_patients)
heart_rate = np.clip(np.round(heart_rate_base), 40, 130).astype(int)

# ECG Abnormality (Age, Previous Heart Problems, Diabetes, BP dependent)
ecg_abnormality_prob_base = 0.01 + (ages - 55) / 100 * 0.4 + previous_heart_problems * 0.5 + (diabetes > 0) * 0.3 + (systolic_bps > 150) * 0.2
ecg_abnormality = (np.random.rand(num_patients) < np.clip(ecg_abnormality_prob_base, 0, 0.7)).astype(int)

# Previous Heart Problems (Age, Diabetes, BP, Family History dependent)
previous_heart_problems_prob_base = 0.005 + (ages - 50) / 80 * 0.5 + (diabetes > 0) * 0.4 + (systolic_bps > 160) * 0.3 + family_history * 0.2
previous_heart_problems = (np.random.rand(num_patients) < np.clip(previous_heart_problems_prob_base, 0, 0.8)).astype(int)

# Kidney Disease (Diabetes, BP, Age, SES dependent)
kidney_disease_prob_base = 0.005 + (diabetes > 0) * 0.5 + (systolic_bps > 150) * 0.3 + (ages > 60) * 0.3 + 0.1 * (1 - ses_proxy)
kidney_disease = (np.random.rand(num_patients) < np.clip(kidney_disease_prob_base, 0, 0.6)).astype(int)

# Heart Disease Risk (Comprehensive risk calculation)
heart_disease_risk_base = (ages - 40) / 40 * 0.3 + (bmis - 25) / 15 * 0.25 + (systolic_bps - 120) / 60 * 0.2 + (ldl_cholesterols - 100) / 70 * 0.15 - (hdl_cholesterols - 40) / 20 * 0.1 + triglycerides / 300 * 0.08 + smoking * 0.3 + (diabetes > 0) * 0.4 + family_history * 0.25 + previous_heart_problems * 0.5 + kidney_disease * 0.35 + ecg_abnormality * 0.4 + (physical_activity < 50) * 0.05 + (diet_quality == 'Poor') * 0.1
heart_disease_prob = np.clip(heart_disease_risk_base, 0, 0.95)
heart_disease = (np.random.rand(num_patients) < heart_disease_prob).astype(int)

# Heart Failure Risk (Dependent on Heart Disease and other factors)
heart_failure_risk_base = (ages - 60) / 30 * 0.5 + (bmis - 30) / 10 * 0.2 + heart_disease * 0.8 + previous_heart_problems * 0.6 + kidney_disease * 0.4 + ecg_abnormality * 0.5 + (diabetes > 0) * 0.3 + (systolic_bps > 160) * 0.15 + (ldl_cholesterols > 150) * 0.08 + anemia * 0.1
heart_failure_prob = np.clip(heart_failure_risk_base, 0, 0.8)
heart_failure = (np.random.rand(num_patients) < heart_failure_prob).astype(int)
heart_failure = np.logical_and(heart_failure, heart_disease).astype(int)

# Medication Usage (More complex logic based on condition severity and other factors)
medication_hypertension = (((systolic_bps > 150) | (diastolic_bps > 95)) * 0.8 +
                           ((systolic_bps > 140) | (diastolic_bps > 90)) * 0.5 * (ages > 60) +
                           (diabetes > 0) * 0.6 +
                           (previous_heart_problems == 1) * 0.4 +
                           (kidney_disease == 1) * 0.3 +
                           (np.random.rand(num_patients) < 0.4)).astype(int)

medication_cholesterol = ((ldl_cholesterols > 160) * 0.9 +
                          (ldl_cholesterols > 130) * 0.6 * (ages > 55) +
                          (hdl_cholesterols < 40) * 0.7 +
                          family_history * 0.5 +
                          (previous_heart_problems == 1) * 0.5 +
                          (diabetes > 0) * 0.4 +
                          (np.random.rand(num_patients) < 0.35)).astype(int)

# Create DataFrame
data = pd.DataFrame({
    'PatientID': np.arange(1, num_patients + 1),
    'Age': ages,
    'Sex': sexes,
    'BMI': bmis,
    'SystolicBP': systolic_bps,
    'DiastolicBP': diastolic_bps,
    'Cholesterol': total_cholesterols,
    'HDL_Cholesterol': hdl_cholesterols,
    'LDL_Cholesterol': ldl_cholesterols,
    'Triglycerides': triglycerides,
    'Smoking': smoking,
    'Diabetes': diabetes,
    'FamilyHistory': family_history,
    'PhysicalActivity': physical_activity,
    'AlcoholConsumption': alcohol_consumption,
    'DietQuality': diet_quality,
    'PreviousHeartProblems': previous_heart_problems,
    'KidneyDisease': kidney_disease,
    'Anemia': anemia,
    'StressLevel': stress_level,
    'SleepDuration': sleep_duration,
    'HeartRate': heart_rate,
    'ECG_Abnormality': ecg_abnormality,
    'Medication_Hypertension': medication_hypertension,
    'Medication_Cholesterol': medication_cholesterol,
    'Heart Failure': heart_failure,
    'Heart Disease': heart_disease
})

# Save to CSV
data.to_csv('realistic_synthetic_heart_data_enhancedv4.csv', index=False)
print(f"Generated an enhanced realistic synthetic dataset of {num_patients} rows and saved it to 'realistic_synthetic_heart_data_enhanced.csv'")

Generated an enhanced realistic synthetic dataset of 250000 rows and saved it to 'realistic_synthetic_heart_data_enhanced.csv'


In [23]:
import pandas as pd
import numpy as np
from scipy.stats import truncnorm

np.random.seed(42)
num_patients = 250000

# Realistic Age Distribution (with slight skew towards older ages)
ages = np.clip(np.round(np.random.beta(a=2, b=3, size=num_patients) * 77 + 18), 18, 95).astype(int)

# Sex
sexes = np.random.choice(['Male', 'Female'], num_patients, p=[0.51, 0.49])
is_male = (sexes == 'Male').astype(int)

# Realistic Mean BMI per Age Group and Sex
def get_mean_bmi(age, sex):
    if sex == 'Male':
        if 18 <= age <= 24:
            return 23.0
        elif 25 <= age <= 34:
            return 25.5
        elif 35 <= age <= 44:
            return 27.5
        elif 45 <= age <= 54:
            return 28.5
        elif 55 <= age <= 64:
            return 28.0
        elif 65 <= age <= 74:
            return 27.5
        else:  # 75+
            return 26.5
    else:  # Female (Slightly lower means)
        if 18 <= age <= 24:
            return 21.5
        elif 25 <= age <= 34:
            return 24.5
        elif 35 <= age <= 44:
            return 26.5
        elif 45 <= age <= 54:
            return 27.0
        elif 55 <= age <= 64:
            return 26.5
        elif 65 <= age <= 74:
            return 26.0
        else:  # 75+
            return 25.0

mean_bmis_age_sex = np.array([get_mean_bmi(ages[i], sexes[i]) for i in range(num_patients)])

# Realistic BMI Distribution per Age Group and Sex (using truncated normal)
std_bmi = 4
lower_bmi = 15
upper_bmi = 55

bmis = np.zeros(num_patients)
for i in range(num_patients):
    mean_bmi = mean_bmis_age_sex[i]
    a_bmi, b_bmi = (lower_bmi - mean_bmi) / std_bmi, (upper_bmi - mean_bmi) / std_bmi
    bmis[i] = np.round(truncnorm.rvs(a_bmi, b_bmi, loc=mean_bmi, scale=std_bmi), 1)
bmis = np.clip(bmis, lower_bmi, upper_bmi)

# Enhanced Blood Pressure (Non-linear BMI effect, interaction with age)
systolic_bp_base = 100 + 0.5 * ages + 0.01 * ages**2 + (0.7 * bmis + 0.005 * bmis**2) + is_male * 5 + np.random.normal(0, 10, num_patients)
systolic_bps = np.clip(np.round(systolic_bp_base), 90, 250).astype(int)
diastolic_bp_base = 60 + 0.3 * ages + (0.4 * bmis + 0.003 * bmis**2) + is_male * 2 + np.random.normal(0, 7, num_patients)
diastolic_bps = np.clip(np.round(diastolic_bp_base), 50, 150).astype(int)

# Alcohol Consumption (More realistic probabilities based on age and sex)
alcohol_options = ['None', 'Light', 'Moderate', 'Heavy']
alcohol_probs = np.zeros((num_patients, len(alcohol_options)))
for i in range(num_patients):
    age_factor = (ages[i] - 40) / 30
    sex_factor = 0.1 if sexes[i] == 'Male' else -0.05
    base_probs = np.array([0.4, 0.35, 0.2, 0.05])
    adjustment = np.array([-0.1 - sex_factor, -0.05 + sex_factor, 0.1 + age_factor, 0.05 + age_factor])
    final_probs = np.clip(base_probs + adjustment, 0.01, 0.98)
    alcohol_probs[i] = final_probs / np.sum(final_probs)
alcohol_consumption = np.array([np.random.choice(alcohol_options, p=prob) for prob in alcohol_probs])

# Diet Quality (Influenced by SES and age)
diet_options = ['Poor', 'Fair', 'Good', 'Excellent']
ses_proxy = np.random.beta(a=1.5, b=3.5, size=num_patients) # Skewed towards lower SES
diet_probs = np.zeros((num_patients, len(diet_options)))
for i in range(num_patients):
    age_factor = (ages[i] - 50) / 40
    base_probs = np.array([0.35, 0.4, 0.2, 0.05])
    adjustment = np.array([0.05 * (1 - ses_proxy[i]) + 0.05 * age_factor,
                             -0.025 * (1 - ses_proxy[i]) - 0.02 * age_factor,
                             -0.015 * ses_proxy[i] - 0.01 * age_factor,
                             -0.01 * ses_proxy[i] + 0.03 * age_factor])
    final_probs = np.clip(base_probs + adjustment, 0.01, 0.98)
    diet_probs[i] = final_probs / np.sum(final_probs)
diet_quality = np.array([np.random.choice(diet_options, p=prob) for prob in diet_probs])
# Family History (Age-dependent)
family_history_prob = 0.2 + 0.3 * (ages / 95)
family_history = (np.random.rand(num_patients) < family_history_prob).astype(int)

# Enhanced Cholesterol (Non-linear, interactions with diet, BMI, family history)
total_cholesterol_base = 160 + 0.6 * ages + 0.008 * ages**2 + (diet_quality == 'Poor') * 30 - (diet_quality == 'Excellent') * 15 + family_history * 20 + 0.3 * bmis + 0.002 * bmis**2 + np.random.normal(0, 25, num_patients)
total_cholesterols = np.clip(np.round(total_cholesterol_base), 100, 450).astype(int)
hdl_cholesterol_base = 60 - 0.2 * ages - (diet_quality == 'Poor') * 12 + (diet_quality == 'Excellent') * 10 - family_history * 8 - 0.15 * bmis + is_male * 5 + np.random.normal(0, 10, num_patients)
hdl_cholesterols = np.clip(np.round(hdl_cholesterols), 20, 120).astype(int)
triglyceride_base = 70 + 0.8 * ages + (diet_quality == 'Poor') * 50 + (alcohol_consumption == 'Heavy') * 40 - (diet_quality == 'Excellent') * 20 + 0.5 * bmis + np.random.normal(0, 40, num_patients)
triglycerides = np.clip(np.round(triglyceride_base), 50, 600).astype(int)
ldl_cholesterols = np.clip(np.round(total_cholesterols - hdl_cholesterols - (triglycerides / 5)), 30, 350).astype(int)
ldl_cholesterols[ldl_cholesterols < 0] = 30

# Physical Activity (Non-linear with BMI and age, SES influence)
physical_activity_base = 200 - 2 * bmis + 0.02 * bmis**2 - 1.2 * ages + 0.01 * ages**2 + 50 * ses_proxy + np.random.normal(0, 60, num_patients)
physical_activity = np.clip(np.round(physical_activity_base), 0, 400).astype(int)

# Smoking (Age, SES, Alcohol, Diet dependent probability)
smoking_prob_base = 0.03 + 0.4 * np.exp(-((ages - 30) ** 2) / (2 * 15 ** 2)) + (diet_quality == 'Poor') * 0.15 + (alcohol_consumption == 'Heavy') * 0.1 - (diet_quality == 'Excellent') * 0.08 - 0.1 * ses_proxy
smoking_prob = np.clip(smoking_prob_base, 0, 0.8)
smoking = (np.random.rand(num_patients) < smoking_prob).astype(int)

# Diabetes (Type 1 and Type 2 with different age dependencies and risk factors)
diabetes = np.zeros(num_patients, dtype=int)
diabetes_prob_type2_base = 0.002 + (ages - 40) / 100 * 0.4 + (bmis - 25) / 10 * 0.5 + family_history * 0.15 + (diet_quality == 'Poor') * 0.2 - (physical_activity > 150) * 0.1
has_diabetes_type2 = (np.random.rand(num_patients) < np.clip(diabetes_prob_type2_base, 0, 0.7))
diabetes_prob_type1_base = 0.005 + np.exp(-((ages - 15) ** 2) / (2 * 8 ** 2)) * 0.08 + family_history * 0.03
has_diabetes_type1 = (np.random.rand(num_patients) < np.clip(diabetes_prob_type1_base, 0, 0.2))
diabetes[has_diabetes_type2] = 2 # Type 2
diabetes[has_diabetes_type1 & ~has_diabetes_type2] = 1 # Type 1

# Anemia (Age and Kidney Disease dependent)
anemia_prob_base = 0.03 + kidney_disease * 0.5 + (ages > 60) * 0.12 - (diet_quality == 'Excellent') * 0.03
anemia = (np.random.rand(num_patients) < np.clip(anemia_prob_base, 0, 0.5)).astype(int)

# Sleep Duration (Stress, Age, Physical Activity, SES dependent)
sleep_duration_base = 7.5 - 0.3 * stress_level - 0.01 * ages + 0.0001 * ages**2 + 0.2 * (physical_activity > 100) - 0.1 * (1 - ses_proxy) + np.random.normal(0, 1.5, num_patients)
sleep_duration = np.clip(np.round(sleep_duration_base), 4, 12).astype(int)

# Stress Level (Sleep, SES, Physical Activity dependent)
stress_level_base = 6 + 0.8 * (1 - ses_proxy) - 0.2 * sleep_duration - 0.005 * physical_activity + np.random.normal(0, 2.5, num_patients)
stress_level = np.clip(np.round(stress_level_base), 1, 10).astype(int)

# Heart Rate (Age, Stress, Physical Activity, BMI dependent)
heart_rate_base = 75 + 0.15 * ages + 0.3 * stress_level - 0.08 * physical_activity + 0.2 * bmis + np.random.normal(0, 12, num_patients)
heart_rate = np.clip(np.round(heart_rate_base), 40, 130).astype(int)

# ECG Abnormality (Age, Previous Heart Problems, Diabetes, BP dependent)
ecg_abnormality_prob_base = 0.01 + (ages - 55) / 100 * 0.4 + previous_heart_problems * 0.5 + (diabetes > 0) * 0.3 + (systolic_bps > 150) * 0.2
ecg_abnormality = (np.random.rand(num_patients) < np.clip(ecg_abnormality_prob_base, 0, 0.7)).astype(int)

# Previous Heart Problems (Age, Diabetes, BP, Family History dependent)
previous_heart_problems_prob_base = 0.005 + (ages - 50) / 80 * 0.5 + (diabetes > 0) * 0.4 + (systolic_bps > 160) * 0.3 + family_history * 0.2
previous_heart_problems = (np.random.rand(num_patients) < np.clip(previous_heart_problems_prob_base, 0, 0.8)).astype(int)

# Kidney Disease (Diabetes, BP, Age, SES dependent)
kidney_disease_prob_base = 0.005 + (diabetes > 0) * 0.5 + (systolic_bps > 150) * 0.3 + (ages > 60) * 0.3 + 0.1 * (1 - ses_proxy)
kidney_disease = (np.random.rand(num_patients) < np.clip(kidney_disease_prob_base, 0, 0.6)).astype(int)

# Heart Disease Risk (Comprehensive risk calculation)
heart_disease_risk_base = (ages - 40) / 40 * 0.3 + (bmis - 25) / 15 * 0.25 + (systolic_bps - 120) / 60 * 0.2 + (ldl_cholesterols - 100) / 70 * 0.15 - (hdl_cholesterols - 40) / 20 * 0.1 + triglycerides / 300 * 0.08 + smoking * 0.3 + (diabetes > 0) * 0.4 + family_history * 0.25 + previous_heart_problems * 0.5 + kidney_disease * 0.35 + ecg_abnormality * 0.4 + (physical_activity < 50) * 0.05 + (diet_quality == 'Poor') * 0.1
heart_disease_prob = np.clip(heart_disease_risk_base, 0, 0.95)
heart_disease = (np.random.rand(num_patients) < heart_disease_prob).astype(int)

# Heart Failure Risk (Dependent on Heart Disease and other factors)
heart_failure_risk_base = (ages - 60) / 30 * 0.5 + (bmis - 30) / 10 * 0.2 + heart_disease * 0.8 + previous_heart_problems * 0.6 + kidney_disease * 0.4 + ecg_abnormality * 0.5 + (diabetes > 0) * 0.3 + (systolic_bps > 160) * 0.15 + (ldl_cholesterols > 150) * 0.08 + anemia * 0.1
heart_failure_prob = np.clip(heart_failure_risk_base, 0, 0.8)
heart_failure = np.logical_and(heart_failure, heart_disease).astype(int)

# Medication Usage (More complex logic based on condition severity and other factors)
medication_hypertension = (((systolic_bps > 150) | (diastolic_bps > 95)) * 0.8 +
                           ((systolic_bps > 140) | (diastolic_bps > 90)) * 0.5 * (ages > 60) +
                           (diabetes > 0) * 0.6 +
                           (previous_heart_problems == 1) * 0.4 +
                           (kidney_disease == 1) * 0.3 +
                           (np.random.rand(num_patients) < 0.4)).astype(int)

medication_cholesterol = ((ldl_cholesterols > 160) * 0.9 +
                          (ldl_cholesterols > 130) * 0.6 * (ages > 55) +
                          (hdl_cholesterols < 40) * 0.7 +
                          family_history * 0.5 +
                          (previous_heart_problems == 1) * 0.5 +
                          (diabetes > 0) * 0.4 +
                          (np.random.rand(num_patients) < 0.35)).astype(int)

# Create DataFrame
data = pd.DataFrame({
    'PatientID': np.arange(1, num_patients + 1),
    'Age': ages,
    'Sex': sexes,
    'BMI': bmis,
    'SystolicBP': systolic_bps,
    'DiastolicBP': diastolic_bps,
    'Cholesterol': total_cholesterols,
    'HDL_Cholesterol': hdl_cholesterols,
    'LDL_Cholesterol': ldl_cholesterols,
    'Triglycerides': triglycerides,
    'Smoking': smoking,
    'Diabetes': diabetes,
    'FamilyHistory': family_history,
    'PhysicalActivity': physical_activity,
    'AlcoholConsumption': alcohol_consumption,
    'DietQuality': diet_quality,
    'PreviousHeartProblems': previous_heart_problems,
    'KidneyDisease': kidney_disease,
    'Anemia': anemia,
    'StressLevel': stress_level,
    'SleepDuration': sleep_duration,
    'HeartRate': heart_rate,
    'ECG_Abnormality': ecg_abnormality,
    'Medication_Hypertension': medication_hypertension,
    'Medication_Cholesterol': medication_cholesterol,
    'Heart Failure': heart_failure,
    'Heart Disease': heart_disease
})

# Save to CSV
data.to_csv('realistic_synthetic_heart_data_enhancedv5.csv', index=False)
print(f"Generated an enhanced realistic synthetic dataset of {num_patients} rows and saved it to 'realistic_synthetic_heart_data_enhanced.csv'")

Generated an enhanced realistic synthetic dataset of 250000 rows and saved it to 'realistic_synthetic_heart_data_enhanced.csv'


With:

1. Add an "Ethnicity" Column.
2. Modify Baseline Risks Based on Ethnicity.
3. Introduce "Diagnosis Year" Columns for  relevant chronic conditions.
4. Simulate Lab Result Variability.
5. Expand Lab Results with other relevant biomarkers.


In [24]:
import pandas as pd
import numpy as np
from scipy.stats import truncnorm

np.random.seed(42)
num_patients = 250000

# Realistic Age Distribution (with slight skew towards older ages)
ages = np.clip(np.round(np.random.beta(a=2, b=3, size=num_patients) * 77 + 18), 18, 95).astype(int)

# Sex
sexes = np.random.choice(['Male', 'Female'], num_patients, p=[0.51, 0.49])
is_male = (sexes == 'Male').astype(int)

# Ethnicity
ethnicities = np.random.choice(['White', 'Black', 'Hispanic', 'Asian'], num_patients, p=[0.6, 0.15, 0.15, 0.1])

# Realistic Mean BMI per Age Group and Sex
def get_mean_bmi(age, sex):
    if sex == 'Male':
        if 18 <= age <= 24: return 23.0
        elif 25 <= age <= 34: return 25.5
        elif 35 <= age <= 44: return 27.5
        elif 45 <= age <= 54: return 28.5
        elif 55 <= age <= 64: return 28.0
        elif 65 <= age <= 74: return 27.5
        else: return 26.5
    else:
        if 18 <= age <= 24: return 21.5
        elif 25 <= age <= 34: return 24.5
        elif 35 <= age <= 44: return 26.5
        elif 45 <= age <= 54: return 27.0
        elif 55 <= age <= 64: return 26.5
        elif 65 <= age <= 74: return 26.0
        else: return 25.0

mean_bmis_age_sex = np.array([get_mean_bmi(ages[i], sexes[i]) for i in range(num_patients)])

# Realistic BMI Distribution per Age Group and Sex (using truncated normal)
std_bmi = 4
lower_bmi = 15
upper_bmi = 55

bmis = np.zeros(num_patients)
for i in range(num_patients):
    mean_bmi = mean_bmis_age_sex[i]
    a_bmi, b_bmi = (lower_bmi - mean_bmi) / std_bmi, (upper_bmi - mean_bmi) / std_bmi
    bmis[i] = np.round(truncnorm.rvs(a_bmi, b_bmi, loc=mean_bmi, scale=std_bmi), 1)
bmis = np.clip(bmis, lower_bmi, upper_bmi)

# Enhanced Blood Pressure (Non-linear BMI effect, interaction with age, ethnicity adjustment)
systolic_bp_base = 100 + 0.5 * ages + 0.01 * ages**2 + (0.7 * bmis + 0.005 * bmis**2) + is_male * 5
systolic_bp_base += np.select(
    [ethnicities == 'Black', ethnicities == 'Asian', ethnicities == 'Hispanic'],
    [3, -1, 1],
    default=0
) + np.random.normal(0, 10, num_patients)
systolic_bps = np.clip(np.round(systolic_bp_base), 90, 250).astype(int)

diastolic_bp_base = 60 + 0.3 * ages + (0.4 * bmis + 0.003 * bmis**2) + is_male * 2
diastolic_bp_base += np.select(
    [ethnicities == 'Black', ethnicities == 'Asian', ethnicities == 'Hispanic'],
    [2, -0.5, 0.5],
    default=0
) + np.random.normal(0, 7, num_patients)
diastolic_bps = np.clip(np.round(diastolic_bp_base), 50, 150).astype(int)

# Alcohol Consumption (More realistic probabilities based on age and sex)
alcohol_options = ['None', 'Light', 'Moderate', 'Heavy']
alcohol_probs = np.zeros((num_patients, len(alcohol_options)))
for i in range(num_patients):
    age_factor = (ages[i] - 40) / 30
    sex_factor = 0.1 if sexes[i] == 'Male' else -0.05
    base_probs = np.array([0.4, 0.35, 0.2, 0.05])
    adjustment = np.array([-0.1 - sex_factor, -0.05 + sex_factor, 0.1 + age_factor, 0.05 + age_factor])
    final_probs = np.clip(base_probs + adjustment, 0.01, 0.98)
    alcohol_probs[i] = final_probs / np.sum(final_probs)
alcohol_consumption = np.array([np.random.choice(alcohol_options, p=prob) for prob in alcohol_probs])

# Diet Quality (Influenced by SES and age)
diet_options = ['Poor', 'Fair', 'Good', 'Excellent']
ses_proxy = np.random.beta(a=1.5, b=3.5, size=num_patients) # Skewed towards lower SES
diet_probs = np.zeros((num_patients, len(diet_options)))
for i in range(num_patients):
    age_factor = (ages[i] - 50) / 40
    base_probs = np.array([0.35, 0.4, 0.2, 0.05])
    adjustment = np.array([
        0.05 * (1 - ses_proxy[i]) + 0.05 * age_factor,
        -0.025 * (1 - ses_proxy[i]) - 0.02 * age_factor,
        -0.015 * ses_proxy[i] - 0.01 * age_factor,
        -0.01 * ses_proxy[i] + 0.03 * age_factor
    ])
    final_probs = np.clip(base_probs + adjustment, 0.01, 0.98)
    diet_probs[i] = final_probs / np.sum(final_probs)
diet_quality = np.array([np.random.choice(diet_options, p=prob) for prob in diet_probs])

# Family History (Age and Ethnicity dependent)
family_history_prob_base = 0.2 + 0.3 * (ages / 95)
family_history_prob_base += np.select(
    [ethnicities == 'Black', ethnicities == 'Asian', ethnicities == 'Hispanic'],
    [0.05, -0.02, 0.03],
    default=0
)
family_history_prob = np.clip(family_history_prob_base, 0, 0.7)
family_history = (np.random.rand(num_patients) < family_history_prob).astype(int)

# Enhanced Cholesterol (Non-linear, interactions with diet, BMI, family history, ethnicity)
total_cholesterol_base = 160 + 0.6 * ages + 0.008 * ages**2 + \
                         (diet_quality == 'Poor') * 30 - \
                         (diet_quality == 'Excellent') * 15 + \
                         family_history * 20 + 0.3 * bmis + 0.002 * bmis**2
total_cholesterol_base += np.select(
    [ethnicities == 'Black', ethnicities == 'Asian', ethnicities == 'Hispanic'],
    [5, -3, 2],
    default=0
) + np.random.normal(0, 25, num_patients)
total_cholesterols = np.clip(np.round(total_cholesterol_base), 100, 450).astype(int)

hdl_cholesterol_base = 60 - 0.2 * ages - \
                       (diet_quality == 'Poor') * 12 + \
                       (diet_quality == 'Excellent') * 10 - \
                       family_history * 8 - 0.15 * bmis + is_male * 5
hdl_cholesterol_base += np.select(
    [ethnicities == 'Black', ethnicities == 'Asian', ethnicities == 'Hispanic'],
    [-2, 4, -1],
    default=0
) + np.random.normal(0, 10, num_patients)
hdl_cholesterols = np.clip(np.round(hdl_cholesterol_base), 20, 120).astype(int)

triglyceride_base = 70 + 0.8 * ages + \
                    (diet_quality == 'Poor') * 50 + \
                    (alcohol_consumption == 'Heavy') * 40 - \
                    (diet_quality == 'Excellent') * 20 + 0.5 * bmis
triglyceride_base += np.select(
    [ethnicities == 'Asian', ethnicities == 'Hispanic'],
    [8, 5],
    default=0
) + np.random.normal(0, 40, num_patients)
triglycerides = np.clip(np.round(triglyceride_base), 50, 600).astype(int)

ldl_cholesterols = np.clip(np.round(total_cholesterols - hdl_cholesterols - (triglycerides / 5)), 30, 350).astype(int)
ldl_cholesterols[ldl_cholesterols < 0] = 30

# Physical Activity (Non-linear with BMI and age, SES influence)
physical_activity_base = 200 - 2 * bmis + 0.02 * bmis**2 - 1.2 * ages + \
                         0.01 * ages**2 + 50 * ses_proxy + np.random.normal(0, 60, num_patients)
physical_activity = np.clip(np.round(physical_activity_base), 0, 400).astype(int)

# Smoking (Age, SES, Alcohol, Diet, Ethnicity dependent probability)
smoking_prob_base = 0.03 + 0.4 * np.exp(-((ages - 30) ** 2) / (2 * 15 ** 2)) + \
                    (diet_quality == 'Poor') * 0.15 + \
                    (alcohol_consumption == 'Heavy') * 0.1 - \
                    (diet_quality == 'Excellent') * 0.08 - 0.1 * ses_proxy
smoking_prob_base += np.select(
    [ethnicities == 'Black', ethnicities == 'Asian'],
    [0.03, -0.02],
    default=0
)
smoking_prob = np.clip(smoking_prob_base, 0, 0.8)
smoking = (np.random.rand(num_patients) < smoking_prob).astype(int)

# Diabetes (Type 1 and Type 2 with different age/ethnicity dependencies and risk factors)
diabetes = np.zeros(num_patients, dtype=int)
diabetes_prob_type2_base = 0.002 + (ages - 40) / 100 * 0.4 + (bmis - 25) / 10 * 0.5 + \
                            family_history * 0.15 + (diet_quality == 'Poor') * 0.2 - \
                            (physical_activity > 150) * 0.1
diabetes_prob_type2_base += np.select(
    [ethnicities == 'Black', ethnicities == 'Hispanic', ethnicities == 'Asian'],
    [0.04, 0.03, 0.02],
    default=0
)
has_diabetes_type2 = (np.random.rand(num_patients) < np.clip(diabetes_prob_type2_base, 0, 0.7))

diabetes_prob_type1_base = 0.005 + np.exp(-((ages - 15) ** 2) / (2 * 8 ** 2)) * 0.08 + family_history * 0.03
has_diabetes_type1 = (np.random.rand(num_patients) < np.clip(diabetes_prob_type1_base, 0, 0.2))

diabetes[has_diabetes_type2] = 2 # Type 2
diabetes[has_diabetes_type1 & ~has_diabetes_type2] = 1 # Type 1

# Anemia (Age, Kidney Disease, Ethnicity dependent)
kidney_disease = (np.random.rand(num_patients) < 0.1).astype(int) # Placeholder for kidney disease
anemia_prob_base = 0.03 + kidney_disease * 0.5 + (ages > 60) * 0.12 - (diet_quality == 'Excellent') * 0.03
anemia_prob_base += np.select(
    [ethnicities == 'Black'],
    [0.06],
    default=0
)
anemia = (np.random.rand(num_patients) < np.clip(anemia_prob_base, 0, 0.5)).astype(int)

# Sleep Duration (Stress, Age, Physical Activity, SES dependent)
stress_level = np.random.randint(1, 11, num_patients) # Placeholder for stress level
sleep_duration_base = 7.5 - 0.3 * stress_level - 0.01 * ages + 0.0001 * ages**2 + \
                      0.2 * (physical_activity > 100) - 0.1 * (1 - ses_proxy) + \
                      np.random.normal(0, 1.5, num_patients)
sleep_duration = np.clip(np.round(sleep_duration_base), 4, 12).astype(int)

# Heart Rate (Age, Stress, Physical Activity, BMI dependent)
heart_rate_base = 75 + 0.15 * ages + 0.3 * stress_level - 0.08 * physical_activity + \
                  0.2 * bmis + np.random.normal(0, 12, num_patients)
heart_rate = np.clip(np.round(heart_rate_base), 40, 130).astype(int)

# ECG Abnormality (Age, Previous Heart Problems, Diabetes, BP dependent)
previous_heart_problems = (np.random.rand(num_patients) < 0.05).astype(int) # Placeholder
ecg_abnormality_prob_base = 0.01 + (ages - 55) / 100 * 0.4 + previous_heart_problems * 0.5 + \
                             (diabetes > 0) * 0.3 + (systolic_bps > 150) * 0.2
ecg_abnormality = (np.random.rand(num_patients) < np.clip(ecg_abnormality_prob_base, 0, 0.7)).astype(int)

# Kidney Disease (Diabetes, BP, Age, SES dependent) - More integrated now
kidney_disease_prob_base = 0.005 + (diabetes > 0) * 0.5 + (systolic_bps > 150) * 0.3 + \
                           (ages > 60) * 0.3 + 0.1 * (1 - ses_proxy)
kidney_disease = (np.random.rand(num_patients) < np.clip(kidney_disease_prob_base, 0, 0.6)).astype(int)

# Heart Disease Risk (Comprehensive risk calculation)
heart_disease_risk_base = (ages - 40) / 40 * 0.3 + (bmis - 25) / 15 * 0.25 + \
                          (systolic_bps - 120) / 60 * 0.2 + (ldl_cholesterols - 100) / 70 * 0.15 - \
                          (hdl_cholesterols - 40) / 20 * 0.1 + triglycerides / 300 * 0.08 + \
                          smoking * 0.3 + (diabetes > 0) * 0.4 + family_history * 0.25 + \
                          previous_heart_problems * 0.5 + kidney_disease * 0.35 + \
                          ecg_abnormality * 0.4 + (physical_activity
                                                   < 50) * 0.05 + (diet_quality == 'Poor') * 0.1
heart_disease_prob = np.clip(heart_disease_risk_base, 0, 0.95)
heart_disease = (np.random.rand(num_patients) < heart_disease_prob).astype(int)

# Heart Failure Risk (Dependent on Heart Disease and other factors)
heart_failure_risk_base = (ages - 60) / 30 * 0.5 + (bmis - 30) / 10 * 0.2 + \
                          heart_disease * 0.8 + previous_heart_problems * 0.6 + \
                          kidney_disease * 0.4 + ecg_abnormality * 0.5 + \
                          (diabetes > 0) * 0.3 + (systolic_bps > 160) * 0.15 + \
                          (ldl_cholesterols > 150) * 0.08 + anemia * 0.1
heart_failure_prob = np.clip(heart_failure_risk_base, 0, 0.8)
heart_failure = (np.random.rand(num_patients) < heart_failure_prob).astype(int)
heart_failure = np.logical_and(heart_failure, heart_disease).astype(int)

# Medication Usage (More complex logic based on condition severity and other factors)
medication_hypertension = (((systolic_bps > 150) | (diastolic_bps > 95)) * 0.8 +
                           ((systolic_bps > 140) | (diastolic_bps > 90)) * 0.5 * (ages > 60) +
                           (diabetes > 0) * 0.6 +
                           (previous_heart_problems == 1) * 0.4 +
                           (kidney_disease == 1) * 0.3 +
                           (np.random.rand(num_patients) < 0.4)).astype(int)

medication_cholesterol = ((ldl_cholesterols > 160) * 0.9 +
                          (ldl_cholesterols > 130) * 0.6 * (ages > 55) +
                          (hdl_cholesterols < 40) * 0.7 +
                          family_history * 0.5 +
                          (previous_heart_problems == 1) * 0.5 +
                          (diabetes > 0) * 0.4 +
                          (np.random.rand(num_patients) < 0.35)).astype(int)

# Diagnosis Year (for realism of undiagnosed conditions)
def get_diagnosis_year(has_condition, age):
    if has_condition:
        # Simulate a diagnosis year, with later diagnoses more likely at older ages
        base_year = 2010  # Arbitrary base year
        diagnosis_age = np.clip(np.round(np.random.normal(loc=age, scale=10)), 18, 90)
        return int(base_year + diagnosis_age - 18)
    else:
        return np.nan

hypertension_diagnosed = (systolic_bps > 140) | (diastolic_bps > 90)
diabetes_diagnosed = (diabetes > 0)
high_cholesterol_diagnosed = (ldl_cholesterols > 130) | (total_cholesterols > 200)

hypertension_diagnosis_year = np.array([get_diagnosis_year(h, a) if np.random.rand() < 0.8 else np.nan for h, a in zip(hypertension_diagnosed, ages)])
diabetes_diagnosis_year = np.array([get_diagnosis_year(d, a) if np.random.rand() < 0.7 else np.nan for d, a in zip(diabetes_diagnosed, ages)])
high_cholesterol_diagnosis_year = np.array([get_diagnosis_year(hc, a) if np.random.rand() < 0.75 else np.nan for hc, a in zip(high_cholesterol_diagnosed, ages)])

# Lab Result Variability
def add_variability(base_value, std_dev_fraction=0.05):
    std_dev = abs(base_value * std_dev_fraction)
    return np.round(np.random.normal(loc=base_value, scale=std_dev), 1)

systolic_bps_varied = np.array([add_variability(bp) for bp in systolic_bps])
diastolic_bps_varied = np.array([add_variability(bp) for bp in diastolic_bps])
total_cholesterols_varied = np.array([add_variability(chol) for chol in total_cholesterols])
hdl_cholesterols_varied = np.array([add_variability(hdl) for hdl in hdl_cholesterols])
ldl_cholesterols_varied = np.array([add_variability(ldl) for ldl in ldl_cholesterols])
triglycerides_varied = np.array([add_variability(trig) for trig in triglycerides])

hba1c = np.where(diabetes > 0, np.clip(np.round(np.random.normal(loc=6.5 + 1.5 * (diabetes == 2), scale=1.2), 1), 4, 14), np.clip(np.round(np.random.normal(loc=5.2, scale=0.4), 1), 4, 6.5))
hba1c_varied = np.array([add_variability(h) for h in hba1c])

creatinine_base = np.clip(np.round(0.8 + 0.01 * ages + kidney_disease * 0.5 + np.random.normal(0, 0.2), 2), 0.5, 3.0)
creatinine_varied = np.array([add_variability(c) for c in creatinine_base])

alt_base = np.clip(np.round(15 + 0.2 * bmis + (alcohol_consumption == 'Heavy') * 20 + np.random.normal(0, 10), 0), 5, 150)
alt_varied = np.array([add_variability(a) for a in alt_base])

# Create DataFrame
data = pd.DataFrame({
    'PatientID': np.arange(1, num_patients + 1),
    'Age': ages,
    'Sex': sexes,
    'Ethnicity': ethnicities,
    'BMI': bmis,
    'SystolicBP': systolic_bps_varied,  # Using varied lab results
    'DiastolicBP': diastolic_bps_varied, # Using varied lab results
    'Cholesterol': total_cholesterols_varied, # Using varied lab results
    'HDL_Cholesterol': hdl_cholesterols_varied, # Using varied lab results
    'LDL_Cholesterol': ldl_cholesterols_varied, # Using varied lab results
    'Triglycerides': triglycerides_varied, # Using varied lab results
    'Smoking': smoking,
    'Diabetes': diabetes,
    'FamilyHistory': family_history,
    'PhysicalActivity': physical_activity,
    'AlcoholConsumption': alcohol_consumption,
    'DietQuality': diet_quality,
    'PreviousHeartProblems': previous_heart_problems,
    'KidneyDisease': kidney_disease,
    'Anemia': anemia,
    'StressLevel': stress_level,
    'SleepDuration': sleep_duration,
    'HeartRate': heart_rate,
    'ECG_Abnormality': ecg_abnormality,
    'Medication_Hypertension': medication_hypertension,
    'Medication_Cholesterol': medication_cholesterol,
    'Heart Failure': heart_failure,
    'Heart Disease': heart_disease,
    'HypertensionDiagnosisYear': hypertension_diagnosis_year,
    'DiabetesDiagnosisYear': diabetes_diagnosis_year,
    'HighCholesterolDiagnosisYear': high_cholesterol_diagnosis_year,
    'HbA1c': hba1c_varied,
    'Creatinine': creatinine_varied,
    'ALT': alt_varied
})

# Save to CSV
data.to_csv('realistic_synthetic_heart_data_enhancedv7.csv', index=False)
print(f"Generated an enhanced realistic synthetic dataset of {num_patients} rows and saved it to 'realistic_synthetic_heart_data_enhancedv7.csv'")

Generated an enhanced realistic synthetic dataset of 250000 rows and saved it to 'realistic_synthetic_heart_data_enhancedv7.csv'
